In [3]:
import pandas as pd
import re
import os

# =============================================================================
# REPLACEMENT DICTIONARIES
# =============================================================================
# Each dictionary is applied at a different stage of variable name creation.
# All replacements are case-sensitive unless noted.
#
# HOW TO ADD:
#   "find this exact string": "replace with this"
#
# Keys are matched longest-first to avoid partial-word collisions.
# Empty string "" as a value means the key is simply deleted.
# =============================================================================


# -----------------------------------------------------------------------------
# STAGE 1 — Applied to raw Label text (before any other cleaning)
#
# Use this for human-readable label text transformations, e.g. expanding
# abbreviations or normalizing phrasing before the label becomes a varname.
# Matches are case-sensitive and applied to the original Label column.
#
# Example use cases:
#   Normalize punctuation:         "In the Past 12 Months" → ""
#   Standardize category names:    "Allocation Rate" → "Allocation/Imputation Rate ..."
# -----------------------------------------------------------------------------
LABEL_TEXT_REPLACEMENTS = {
    " in the Past 12 Months": "",                               # redundant for 1yr estimates
    "Allocation Rate": "Allocation/Imputation Rate (Imputation Rate of)",
    "Allocation of": "Allocation/Imputation (Imputation Rate of) ",
}


# -----------------------------------------------------------------------------
# STAGE 2 — Applied to label_varname and detail_varname (after lowercasing +
#           symbol-to-underscore conversion, before final cleanup)
#
# Use this for snake_case substring replacements — racial/ethnic group
# shorthands, redundant words, long phrases you want abbreviated, etc.
# All text here is already lowercase with spaces/symbols replaced by "_".
#
# Example use cases:
#   Shorten race/ethnicity labels: "black_or_african_american" → "black"
#   Remove filler words:           "_in_the_past_12_months"   → ""
#   Abbreviate common words:       "estimate" → "est"
# -----------------------------------------------------------------------------
VARNAME_REPLACEMENTS = {
    # Race / ethnicity
    "american_indian_and_alaska_native":          "AIAN",
    "black_or_african_american":                  "black",
    "native_hawaiian_and_other_pacific_islander": "NHPI",
    "some_other_race":                            "other_race",
    "two_or_more_races":                          "multiple_races",
    "not_hispanic_or_latino":                     "nh",
    "hispanic_or_latino":                         "hisp",
    "_alone":                                     "",         # e.g. "white_alone" → "white"

    # Common word abbreviations
    "estimate":                                   "", #default is that it's estimate
    "percent":                                    "perc",
    "total":                                      "", #default is total
    "year":                                       "yr",

    # Redundant phrases 
    "_in_the_past_12_months":                     "", #(1yr ACS — these are always implied)
    "_for_the_civilian_employed_population_16_yrs_and_over": "_civ_emp_pop16plus",
    "_for_the_civilian_population_18_yrs_and_over":          "_civ_pop18plus",
    "_for_the_population_16_yrs_and_over":                   "_pop16plus",
    "_for_the_population_15_yrs_and_over":                   "_pop15plus",
    "_for_the_foreign_born_population":                      "_foreign_born",
    "_for_current_residence_in_the_united_states":           "",   # implied
    "_in_the_united_states":                                 "",   # implied
    "_in_puerto_rico":                                       "_pr",

    # Symbols that slip through
    "$":                                          "",
    "_to_":                                       "_",        # e.g. age ranges "25_to_34" → "25_34"

    # Reduce unnecessary connectors liky "by"
    "_by_nativity_and_citizenship_status_by_sex": "_nativity_citizenship_sex",
    "_by_nativity_and_citizenship_status":        "_nativity_citizenship",
    "_by_marital_status":                         "_marital_status",
    "_by_age_by_":                               "_age_",
    "_by_age_":                                  "_age_",
    "_by_":                                      "_",   # catch-all, apply last

    # Allocation/imputation shorthand (matches post-LABEL_TEXT_REPLACEMENTS output)
    "allocation_imputation_rate":                 "imp_rate",
    "allocation_imputation_of":                   "imp_of",
    "allocation_imputation":                      "imp",

    # Filler words
    "ratio_of_income_poverty_level_of_families": "income_poverty_ratio_families",
    "poverty_status_of_families":                "poverty_families",
    "receipt_of_food_stamps_snap":               "snap_receipt",
    "geographical_mobility_in_the_past_yr":      "geo_mobility",
    "detailed_occupation_for_the":               "occupation_",
    "place_of_birth":                            "pob",
    "_and_or_":                                  "_or_",
    "_and_":                                     "_",  # catch-all, apply last


    # Detail-level redundant phrases:
    "now_married_married_spouse_absent":               "married_spouse_absent",  # deduplicate
    "no_related_children_of_the_householder":          "no_related_children",
    "household_did_not_receive_food_stamps_snap":      "no_snap",
    "income_below_poverty_level":                      "below_poverty",
    "naturalized_u.s._citizen":                        "naturalized_citizen",
    "in_labor_force":                                  "labor_force",
    "with_earnings":                                   "w_earnings",
    "without_social_security_income":                  "no_ss_income",
    "with_ssi_and_or_cash_public_assistance_income":   "w_ssi_cash_assist",
    "with_ssi_and_cash_public_assistance_income":      "w_ssi_cash_assist",
    "male_householder_no_spouse_present":              "male_hholder_no_spouse",
    "female_householder_no_spouse_present":            "female_hholder_no_spouse",
    "american_indian_tribes_specified":                "AI_tribes",
    "entered_":                                        "entry_",   # e.g. "entry_2000_2009"
    "groups_tallied":                                  "groups",
}
# -----------------------------------------------------------------------------
# STAGE 3 — Applied to both_varname (the combined "label__detail" string)
#           after it has already been through STAGE 1 + STAGE 2 processing.
#
# Use this to clean up patterns that only appear once label and detail are
# joined, e.g. redundant prefixes introduced by a specific table's label.
#
# Example use cases:
#   Strip table-level prefix that adds noise to every variable name:
#       "acs_demographic_and_housing_ests__" → ""
# -----------------------------------------------------------------------------
BOTH_VARNAME_REPLACEMENTS = {
    "acs_demographic_and_housing_ests__": "",   # prefix redundant once variables are in context
    "selected_social_characteristics_in_puerto_rico__": "soc_chars_pr__",
    "selected_social_characteristics_in_the_united_states__": "soc_chars__",
    "selected_housing_characteristics__":              "housing_chars__",
    "selected_economic_characteristics__":             "econ_chars__",
    "population_60_yrs_and_over_in_the_united_states__": "pop60plus__",
}


# =============================================================================
# FILE PATHS  — change filenames here if needed
# =============================================================================
### !!! Keep these in sync with the filenames referenced in App.jsx !!! ###

wd         = os.getcwd()
DATA_DIR   = os.path.join(wd, "public")

INPUT_FILE  = "1yr_raw_variables.csv"
OUTPUT_FILE = "1yr_clean_varnames.csv"
OUTPUT_SHORT = "1yr_clean_varnames_shortened.csv"   # first 10,000 rows for dev/testing

inputFP      = os.path.join(DATA_DIR, INPUT_FILE)
outputFP     = os.path.join(DATA_DIR, OUTPUT_FILE)
outputFPshort = os.path.join(DATA_DIR, OUTPUT_SHORT)

### !!! ################################################################ !!! ###


# =============================================================================
# HELPERS
# =============================================================================

def _apply_dict(text: str, replacements: dict) -> str:
    """Apply a replacement dictionary to a string, longest keys first."""
    for key in sorted(replacements, key=len, reverse=True):
        text = text.replace(key, replacements[key])
    return text


def _to_snake(text: str) -> str:
    """Lowercase a string and convert common separators to underscores."""
    text = text.lower()
    for ch in ("!!", "(", ")", ",", ":", " ", "-", "/"):
        text = text.replace(ch, "_")
    # Collapse runs of underscores and strip leading/trailing ones
    text = re.sub(r"_+", "_", text).strip("_")
    return text


# =============================================================================
# CORE PROCESSING
# =============================================================================

def create_coding_varnames(csv_filepath: str, output_filepath: str = None) -> pd.DataFrame:
    df = pd.read_csv(csv_filepath)
    df["Label"]  = df["Label"].fillna("")
    df["Detail"] = df["Detail"].fillna("")
    df["Label_old"] = df["Label"]   # preserve original for reference

    # ------------------------------------------------------------------
    # Stage 1: clean label text (human-readable, pre-snake-case)
    # ------------------------------------------------------------------
    df["label_clean"] = df["Label"].apply(
        lambda text: _apply_dict(str(text), LABEL_TEXT_REPLACEMENTS)
    )

    # ------------------------------------------------------------------
    # Stage 2: convert to snake_case varnames, then apply VARNAME_REPLACEMENTS
    # ------------------------------------------------------------------
    df["label_varname"] = (
        df["label_clean"]
        .apply(_to_snake)
        .apply(lambda s: _apply_dict(s, VARNAME_REPLACEMENTS))
    )

    df["detail_varname"] = (
        df["Detail"]
        .apply(lambda text: _to_snake(str(text)))
        .apply(lambda s: _apply_dict(s, VARNAME_REPLACEMENTS))
    )

    # ------------------------------------------------------------------
    # Stage 3: combine into both_varname, then apply BOTH_VARNAME_REPLACEMENTS
    # ------------------------------------------------------------------
    df["both_varname"] = (
        (df["label_varname"] + "__" + df["detail_varname"])
        .apply(lambda s: _apply_dict(s, BOTH_VARNAME_REPLACEMENTS))
    )

    if output_filepath:
        df.to_csv(output_filepath, index=False)
        print(f"Saved: {output_filepath}")

    return df


# =============================================================================
# RUN
# =============================================================================

if __name__ == "__main__":
    df_varnames = create_coding_varnames(inputFP, output_filepath=outputFP)
    df_varnames.head(10000).to_csv(outputFPshort, index=False)
    print(df_varnames.head(10).to_string())

Saved: /Users/klj9278/variable-explorer/public/1yr_clean_varnames.csv
            ID                                    Detail       Label Unnamed: 3 Unnamed: 4      Required                               Attributes Limit     Predicate Type   Group   Label_old label_clean label_varname      detail_varname                 both_varname
0       AIANHH                                 Geography                    NaN        NaN  not required                                      NaN     0  (not a predicate)     NaN                                                 geography                  __geography
1         ANRC                                 Geography                    NaN        NaN  not required                                      NaN     0  (not a predicate)     NaN                                                 geography                  __geography
2  B01001_001E                          Estimate!!Total:  Sex by Age        NaN        NaN  not required  B01001_001EA, B01001_001M,